# Prepare GSE245998 + Heemskerk data for MultiVI

This notebook is the local preparation step. It does **not** train MultiVI.

It creates one integration-ready MuData (`.h5mu`) containing:

- paired GSE245998 RNA and consensus-peak ATAC;
- Heemskerk 48–96 h RNA (`bc_*` excluded);
- both Heemskerk 42 h RNA replicates;
- optionally Minn 0–24 h RNA;
- a 10,000-gene, batch-aware HVG ranking computed only from Heemskerk 48–96 h, without subsetting the stored RNA matrix; and
- Macosko cell-cycle membership annotations on the RNA features.

The EC2 runner downloads this prepared file, independently selects the 1,500- and 2,000-HVG tiers, copies the matching cell-cycle gene expression into continuous covariates, trains MultiVI, and saves the models and inferred accessibility.

A `.h5mu` is used instead of `.h5ad` because MultiVI needs distinct RNA and ATAC modalities plus the paired/expression-only labels.


In [1]:
from __future__ import annotations

import json
import os
import re
from pathlib import Path

import anndata as ad
import mudata as md
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"

GSE_H5MU = Path(os.environ.get(
    "GSE245998_MULTIVI_H5MU",
    DATA / "GSE245998/processed/gse245998_multiome_qc.h5mu",
))
GSE_ATAC_H5AD = Path(os.environ.get(
    "GSE245998_ATAC_H5AD",
    DATA / "GSE245998/processed/archr_peak_matrix/gse245998_archr_peak_matrix.h5ad",
))
HEEMSKERK_D2_D10 = ROOT / "heemskerk_data/adata_timeseries_old_48-96h_new_D6-10_filtered_qc.h5ad"
HEEMSKERK_42H_REP1 = ROOT / "heemskerk_data/adata_2020_force_9000.h5ad"
HEEMSKERK_42H_REP2 = ROOT / "heemskerk_data/adata_2021_BMP_contorl.h5ad"
CELL_CYCLE_GENES = ROOT / "heemskerk_data/Macosko_cell_cycle_genes.txt"
MINN_0_24H = ROOT / "minn_data/minn_gastruloid_0-24h_qc_20260213.h5ad"

HVG_POOL_SIZE = int(os.environ.get("MULTIVI_HVG_POOL_SIZE", "10000"))
ATAC_MIN_CELL_FRACTION = float(os.environ.get("MULTIVI_ATAC_MIN_CELL_FRACTION", "0.01"))
INCLUDE_MINN = os.environ.get("INCLUDE_MINN", "0") == "1"
PREPARED_OUTPUT = Path(os.environ.get(
    "MULTIVI_PREPARED_OUTPUT",
    DATA / "GSE245998/processed/gse245998_heemskerk_multivi_ready.h5mu",
))
PREPARED_S3 = os.environ.get(
    "MULTIVI_PREPARED_S3",
    (
        "stan-sequencing-data/processed-active/"
        "gse245998-tbxt-multiome-integration/input/"
        "gse245998_heemskerk_multivi_ready.h5mu"
    ),
).removeprefix("s3://")

print(f"include Minn 0–24 h: {INCLUDE_MINN}")
print(f"HVG ranking pool: {HVG_POOL_SIZE:,}")
print(f"ATAC min-cell fraction: {ATAC_MIN_CELL_FRACTION:.3f}")
print(f"prepared output: {PREPARED_OUTPUT}")
print(f"upload destination: s3://{PREPARED_S3}")


include Minn 0–24 h: False
HVG ranking pool: 10,000
ATAC min-cell fraction: 0.010
prepared output: /home/stan/Git/micropattern/data/GSE245998/processed/gse245998_heemskerk_multivi_ready.h5mu
upload destination: s3://stan-sequencing-data/processed-active/gse245998-tbxt-multiome-integration/input/gse245998_heemskerk_multivi_ready.h5mu


## Input preflight

The GSE245998 input must contain RNA and a non-empty consensus-peak ATAC matrix. The earlier observation-only ATAC QC object is not sufficient for MultiVI.


In [2]:
required = {
    "GSE245998 MuData": GSE_H5MU,
    "GSE245998 ArchR peak matrix": GSE_ATAC_H5AD,
    "Heemskerk 48–96 h": HEEMSKERK_D2_D10,
    "Heemskerk 42 h rep1": HEEMSKERK_42H_REP1,
    "Heemskerk 42 h rep2": HEEMSKERK_42H_REP2,
    "Macosko cell-cycle genes": CELL_CYCLE_GENES,
}
if INCLUDE_MINN:
    required["Minn 0–24 h"] = MINN_0_24H

missing = []
for label, path in required.items():
    exists = path.exists()
    print(f"{label:32s} {str(path):90s} {'OK' if exists else 'MISSING'}")
    if not exists:
        missing.append(path)
if missing:
    raise FileNotFoundError("Stage all inputs listed above before continuing.")


GSE245998 MuData                 /home/stan/Git/micropattern/data/GSE245998/processed/gse245998_multiome_qc.h5mu            OK
GSE245998 ArchR peak matrix      /home/stan/Git/micropattern/data/GSE245998/processed/archr_peak_matrix/gse245998_archr_peak_matrix.h5ad OK
Heemskerk 48–96 h                /home/stan/Git/micropattern/heemskerk_data/adata_timeseries_old_48-96h_new_D6-10_filtered_qc.h5ad OK
Heemskerk 42 h rep1              /home/stan/Git/micropattern/heemskerk_data/adata_2020_force_9000.h5ad                      OK
Heemskerk 42 h rep2              /home/stan/Git/micropattern/heemskerk_data/adata_2021_BMP_contorl.h5ad                     OK
Macosko cell-cycle genes         /home/stan/Git/micropattern/heemskerk_data/Macosko_cell_cycle_genes.txt                    OK


In [3]:
gse = md.read_h5mu(GSE_H5MU)
rna_key = "rna"

if rna_key not in gse.mod:
    raise KeyError(f"Expected RNA modality '{rna_key}'; found {list(gse.mod)}")

atac_gse = ad.read_h5ad(GSE_ATAC_H5AD)
if atac_gse.n_vars == 0:
    raise ValueError("The ArchR consensus-peak matrix has no ATAC regions.")
rna_cells = gse.mod[rna_key].obs_names
if not rna_cells.is_unique or not atac_gse.obs_names.is_unique:
    raise ValueError("GSE245998 RNA and ATAC cell identifiers must be unique.")
if set(rna_cells) != set(atac_gse.obs_names):
    raise ValueError("GSE245998 RNA and ATAC cell identifier sets differ.")
atac_gse = atac_gse[rna_cells].copy()
if not rna_cells.equals(atac_gse.obs_names):
    raise ValueError("Failed to reorder GSE245998 ATAC cells to the RNA axis.")
print(gse)
print(f"ArchR ATAC matrix: {atac_gse.n_obs:,} cells × {atac_gse.n_vars:,} peaks")


/home/stan/Git/micropattern/.venv/lib/python3.12/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/stan/Git/micropattern/.venv/lib/python3.12/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


MuData object with n_obs × n_vars = 21604 × 29598
  obs:	'barcode', 'sample', 'sample_name', 'genotype', 'batch', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'pass_author_rna_qc', 'pass_basic_rna_qc', 'doublet_score', 'predicted_doublet', 'atac_n_peaks_by_counts', 'atac_peak_counts', 'atac_n_fragments', 'atac_nucleosome_signal', 'atac_tss_score', 'pass_atac_qc', 'pass_doublet_qc', 'pass_paired_qc'
  uns:	'provenance'
  2 modalities
    rna:	21604 x 29598
      obs:	'barcode', 'sample', 'sample_name', 'genotype', 'batch', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'pass_author_rna_qc', 'pass_basic_rna_qc', 'doublet_score', 'predicted_doublet', 'atac_n_peaks_by_counts', 'atac_peak_counts', 'atac_n_fragments', 'atac_nucleosome_signal', 'atac_tss_score', 'pass_atac_qc', 'pass_doublet_qc', 'pass_paired_qc'
      var:	'gene_ids', 'feature_types', 'genome'

## Load the RNA-only reference

The `bc_*` labels are the D6–D10 branch of the combined Heemskerk object. They are excluded, leaving 48 h, 72 h, and 96 h cells.


In [4]:
heem_48_96 = sc.read_h5ad(HEEMSKERK_D2_D10)
keep = ~heem_48_96.obs["sample_labels"].astype(str).str.startswith("bc_")
heem_48_96 = heem_48_96[keep].copy()
heem_48_96.obs["timepoint_hours"] = (
    heem_48_96.obs["sample_labels"].astype(str).str.extract(r"^(\d+)h", expand=False).astype(int)
)
heem_48_96.obs["replicate"] = (
    heem_48_96.obs["sample_labels"].astype(str).str.extract(r"-(\d+)$", expand=False)
)

heem_42_parts = []
for replicate, path in {"rep1": HEEMSKERK_42H_REP1, "rep2": HEEMSKERK_42H_REP2}.items():
    x = sc.read_h5ad(path)
    x.obs["sample_labels"] = x.obs["sample_labels"].astype(str) + "_42h"
    x.obs["replicate"] = replicate
    x.obs["timepoint_hours"] = 42
    heem_42_parts.append(x)
heem_42 = ad.concat(heem_42_parts, join="inner", keys=["rep1", "rep2"], index_unique="_")

minn = sc.read_h5ad(MINN_0_24H) if INCLUDE_MINN else None
if minn is not None:
    minn.obs["timepoint_hours"] = (
        minn.obs["sample_labels"].astype(str).str.extract(r"(\d+)h", expand=False).astype(int)
    )

print("Heemskerk 48–96 h:", heem_48_96.shape)
print("Heemskerk 42 h:", heem_42.shape)
if minn is not None:
    print("Minn 0–24 h:", minn.shape)


Heemskerk 48–96 h: (46508, 16519)
Heemskerk 42 h: (10815, 20685)


## Harmonize genes and compute the gold-standard HVG ranking

Genes are matched by version-stripped Ensembl ID. Heemskerk 48–96 h alone defines the batch-aware Seurat-v3 ranking. `subset=False` retains the full common-gene matrix in the prepared artifact.


In [5]:
ENSG_RE = re.compile(r"\((ENSG\d+)(?:\.\d+)?\)$")


def harmonize_genes(x: ad.AnnData) -> ad.AnnData:
    x = x.copy()
    if "gene_ids" in x.var:
        ids = pd.Index(x.var["gene_ids"].astype(str)).str.replace(r"\.\d+$", "", regex=True)
        symbols = (
            x.var["gene_symbol"].astype(str).to_numpy()
            if "gene_symbol" in x.var
            else x.var_names.astype(str).to_numpy()
        )
    else:
        names = x.var_names.astype(str)
        ids = pd.Index([m.group(1) if (m := ENSG_RE.search(name)) else "" for name in names])
        symbols = np.asarray([name.rsplit(" (", 1)[0] for name in names], dtype=object)

    keep = ids.str.startswith("ENSG") & ~ids.duplicated()
    x = x[:, np.asarray(keep)].copy()
    x.var["gene_symbol"] = symbols[np.asarray(keep)]
    x.var_names = ids[keep]
    if not x.var_names.is_unique:
        raise ValueError("Ensembl IDs are not unique after harmonization.")
    return x


rna_gse = harmonize_genes(gse.mod[rna_key])
heem_48_96 = harmonize_genes(heem_48_96)
heem_42 = harmonize_genes(heem_42)
if minn is not None:
    minn = harmonize_genes(minn)

common_genes = set(rna_gse.var_names)
reference_parts = [heem_48_96, heem_42]
if minn is not None:
    reference_parts.append(minn)
for x in reference_parts:
    common_genes &= set(x.var_names)
common_gene_order = [gene for gene in heem_48_96.var_names if gene in common_genes]
print(f"Common genes across all inputs: {len(common_gene_order):,}")
if len(common_gene_order) < HVG_POOL_SIZE:
    raise ValueError(
        f"Only {len(common_gene_order)} common genes; cannot create the requested "
        f"{HVG_POOL_SIZE}-gene HVG ranking."
    )

hvg_reference = heem_48_96[:, common_gene_order].copy()
sc.pp.highly_variable_genes(
    hvg_reference,
    n_top_genes=HVG_POOL_SIZE,
    flavor="seurat_v3",
    batch_key="sample_labels",
    subset=False,
)
for n_hvg in (1500, 2000):
    hvg_reference.var[f"top{n_hvg}"] = False
    exact_top = hvg_reference.var["highly_variable_rank"].nsmallest(n_hvg).index
    hvg_reference.var.loc[exact_top, f"top{n_hvg}"] = True

cc_table = pd.read_csv(CELL_CYCLE_GENES, sep="\t").iloc[:, :5]
cc_by_phase = {
    phase: [str(gene).strip() for gene in genes.dropna() if str(gene).strip()]
    for phase, genes in cc_table.items()
}
symbol_to_phases = {}
for phase, genes in cc_by_phase.items():
    for gene in genes:
        symbol_to_phases.setdefault(gene.upper(), []).append(phase)
hvg_reference.var["macosko_cell_cycle"] = [
    str(symbol).strip().upper() in symbol_to_phases
    for symbol in hvg_reference.var["gene_symbol"]
]
hvg_reference.var["macosko_phases"] = [
    "|".join(symbol_to_phases.get(str(symbol).strip().upper(), []))
    for symbol in hvg_reference.var["gene_symbol"]
]

print(f"Ranked HVGs: {int(hvg_reference.var['highly_variable'].sum()):,}")
for n_hvg in (1500, 2000):
    n_cc = int(
        (
            hvg_reference.var[f"top{n_hvg}"]
            & hvg_reference.var["macosko_cell_cycle"]
        ).sum()
    )
    print(f"top {n_hvg:,}: {n_cc:,} Macosko cell-cycle covariates")


Common genes across all inputs: 14,992
Ranked HVGs: 10,000
top 1,500: 108 Macosko cell-cycle covariates
top 2,000: 133 Macosko cell-cycle covariates


## Assemble the integration-ready MuData

The prepared RNA modality retains all common genes and raw counts in `.X`; the EC2 runner creates its smaller `counts` layer only after selecting an HVG tier. Source observation metadata are preserved with an outer column union, while genes remain restricted to the common inner intersection. Expression-only cells receive zero-filled ATAC rows; `modality_batch` tells MultiVI those zeros are structurally missing rather than observed closed chromatin.


In [6]:
def add_labels(
    x: ad.AnnData,
    *,
    dataset: str,
    source: str,
    modality: str,
    timepoint_hours: int | None = None,
) -> ad.AnnData:
    x = x.copy()
    prefix = f"{dataset}:"
    names = x.obs_names.astype(str)
    already_prefixed = names.str.startswith(prefix)
    if already_prefixed.any() and not already_prefixed.all():
        raise ValueError(f"Only some {dataset} observation names have the dataset prefix.")
    if not already_prefixed.all():
        x.obs_names = pd.Index([f"{prefix}{name}" for name in names])
    x.obs["dataset"] = dataset
    x.obs["source"] = source
    x.obs["modality_batch"] = modality
    if "sample_labels" in x.obs:
        sample = x.obs["sample_labels"].astype(str)
    elif "sample" in x.obs:
        sample = x.obs["sample"].astype(str)
    else:
        sample = pd.Series(dataset, index=x.obs_names, dtype=str)
    x.obs["sample_id"] = dataset + ":" + sample.astype(str)
    if timepoint_hours is not None:
        x.obs["timepoint_hours"] = timepoint_hours
    return x


rna_gse = add_labels(
    rna_gse,
    dataset="gse245998",
    source="gse245998",
    modality="paired",
    timepoint_hours=48,
)
heem_48_96 = add_labels(
    heem_48_96,
    dataset="heemskerk_48_96h",
    source="heemskerk",
    modality="expression",
)
heem_42 = add_labels(
    heem_42,
    dataset="heemskerk_42h",
    source="heemskerk",
    modality="expression",
)
if minn is not None:
    minn = add_labels(
        minn,
        dataset="minn_0_24h",
        source="minn",
        modality="expression",
    )

labeled_parts = [rna_gse, heem_48_96, heem_42]
if minn is not None:
    labeled_parts.append(minn)
obs_all = pd.concat(
    [x.obs for x in labeled_parts],
    axis=0,
    join="outer",
    sort=False,
)
# An outer metadata union introduces missing values. Normalize mixed object
# columns to H5AD/H5Mu-safe nullable booleans or categoricals without turning
# unmeasured QC flags into False.
for column in obs_all.select_dtypes(include="object").columns:
    values = obs_all[column].dropna()
    if values.empty:
        obs_all[column] = pd.Categorical(obs_all[column])
    elif values.map(lambda value: isinstance(value, (bool, np.bool_))).all():
        obs_all[column] = obs_all[column].astype("boolean")
    elif values.map(lambda value: isinstance(value, str)).all():
        obs_all[column] = pd.Categorical(obs_all[column])
    else:
        observed_types = sorted({type(value).__name__ for value in values})
        raise TypeError(
            f"Observation column '{column}' has unsupported mixed values: "
            f"{observed_types}"
        )
remaining_object_columns = obs_all.select_dtypes(include="object").columns.tolist()
if remaining_object_columns:
    raise TypeError(
        f"Observation columns remain non-serializable: {remaining_object_columns}"
    )
if not obs_all.index.is_unique:
    raise ValueError("Observation identifiers are not unique across input datasets.")
rna_all = ad.concat(
    [x[:, common_gene_order].copy() for x in labeled_parts],
    join="inner",
    merge="first",
)
if not rna_all.obs_names.equals(obs_all.index):
    raise ValueError("RNA concatenation changed the expected observation order.")
rna_all.obs = obs_all.loc[rna_all.obs_names].copy()
rna_all.X = sp.csr_matrix(rna_all.X)
rna_all.var = hvg_reference.var.reindex(rna_all.var_names).copy()

required_gse_metadata = {"sample", "sample_name", "genotype", "batch"}
missing_gse_metadata = required_gse_metadata.difference(rna_all.obs.columns)
if missing_gse_metadata:
    raise ValueError(
        f"GSE245998 metadata were lost during concatenation: {sorted(missing_gse_metadata)}"
    )
gse_mask = rna_all.obs["dataset"].astype(str).eq("gse245998")
if rna_all.obs.loc[gse_mask, sorted(required_gse_metadata)].isna().any().any():
    raise ValueError("GSE245998 sample, genotype, or batch metadata contain missing values.")

atac_gse.obs_names = rna_gse.obs_names.copy()
atac_gse.X = sp.csr_matrix(atac_gse.X)
min_atac_cells = max(1, int(np.ceil(atac_gse.n_obs * ATAC_MIN_CELL_FRACTION)))
peak_n_cells = np.asarray((atac_gse.X > 0).sum(axis=0)).ravel()
keep_peaks = peak_n_cells >= min_atac_cells
atac_gse = atac_gse[:, keep_peaks].copy()
atac_gse.var["n_cells"] = peak_n_cells[keep_peaks]
if atac_gse.n_vars == 0:
    raise ValueError("No ATAC regions remain after prevalence filtering.")
print(
    f"ATAC regions retained: {atac_gse.n_vars:,} "
    f"(present in >= {min_atac_cells:,} paired cells)"
)

# The vstack below is valid only when paired GSE cells are the leading RNA block
# in exactly the same order as the rows of the ATAC matrix.
if not rna_all.obs_names.is_unique or not atac_gse.obs_names.is_unique:
    raise ValueError("RNA and ATAC observation identifiers must be unique.")
atac_positions = rna_all.obs_names.get_indexer(atac_gse.obs_names)
if np.any(atac_positions < 0):
    raise ValueError("Some GSE ATAC cells are absent from the assembled RNA axis.")
expected_atac_positions = np.arange(atac_gse.n_obs)
if not np.array_equal(atac_positions, expected_atac_positions):
    raise ValueError(
        "GSE ATAC rows are not aligned to the leading RNA block; "
        "refusing to assemble a mislabeled matrix."
    )

n_rna_only = rna_all.n_obs - atac_gse.n_obs
atac_x = sp.vstack(
    [
        atac_gse.X,
        sp.csr_matrix(
            (n_rna_only, atac_gse.n_vars),
            dtype=atac_gse.X.dtype,
        ),
    ],
    format="csr",
)
atac_all = ad.AnnData(
    X=atac_x,
    obs=rna_all.obs.copy(),
    var=atac_gse.var.copy(),
)

mdata = md.MuData({"rna": rna_all, "atac": atac_all})
mdata.update()
mdata.obs = rna_all.obs.copy()
mdata.uns["preparation"] = {
    "hvg_selection_source": "heemskerk_48_96h",
    "hvg_flavor": "seurat_v3",
    "hvg_batch_key": "sample_labels",
    "hvg_pool_size": HVG_POOL_SIZE,
    "atac_min_cell_fraction": ATAC_MIN_CELL_FRACTION,
    "include_minn": INCLUDE_MINN,
    "cell_cycle_source": CELL_CYCLE_GENES.name,
}
print(mdata)
print(mdata.obs["dataset"].value_counts())
print(mdata.obs.loc[gse_mask, "genotype"].value_counts())
print(f"Preserved observation columns: {len(mdata.obs.columns):,}")


ATAC regions retained: 177,131 (present in >= 217 paired cells)
MuData object with n_obs × n_vars = 78927 × 192123
  obs:	'barcode', 'sample', 'sample_name', 'genotype', 'batch', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'pass_author_rna_qc', 'pass_basic_rna_qc', 'doublet_score', 'predicted_doublet', 'atac_n_peaks_by_counts', 'atac_peak_counts', 'atac_n_fragments', 'atac_nucleosome_signal', 'atac_tss_score', 'pass_atac_qc', 'pass_doublet_qc', 'pass_paired_qc', 'dataset', 'source', 'modality_batch', 'sample_id', 'timepoint_hours', 'sample_labels', 'replicate'
  uns:	'preparation'
  2 modalities
    rna:	78927 x 14992
      obs:	'barcode', 'sample', 'sample_name', 'genotype', 'batch', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'pass_author_rna_qc', 'pass_basic_rna_qc', 'doublet_score', 'predicted_doublet', 'atac_n_peaks_by_counts', 'atac_peak_counts

/home/stan/Git/micropattern/.venv/lib/python3.12/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/stan/Git/micropattern/.venv/lib/python3.12/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)
/home/stan/Git/micropattern/.venv/lib/python3.12/site-packages/mudata/_core/mudata.py:1403: FutureWarning: Fro

## Save locally, inspect, then upload

Run the save cell first. After checking the printed dimensions and metadata, run the upload cell when you are ready.


In [7]:
PREPARED_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
mdata.write_h5mu(PREPARED_OUTPUT)
print(f"Saved {PREPARED_OUTPUT} ({PREPARED_OUTPUT.stat().st_size / 2**30:.2f} GiB)")


Saved /home/stan/Git/micropattern/data/GSE245998/processed/gse245998_heemskerk_multivi_ready.h5mu (6.18 GiB)


/home/stan/Git/micropattern/.venv/lib/python3.12/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/stan/Git/micropattern/.venv/lib/python3.12/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


In [8]:
# Deliberately separate from saving so upload remains an explicit user action.
import s3fs

s3 = s3fs.S3FileSystem()
print(f"Uploading {PREPARED_OUTPUT} -> s3://{PREPARED_S3}")
s3.put_file(str(PREPARED_OUTPUT), PREPARED_S3)
remote_size = s3.info(PREPARED_S3)["size"]
if remote_size != PREPARED_OUTPUT.stat().st_size:
    raise IOError("Uploaded object size does not match the local prepared file.")
print(f"Upload complete: s3://{PREPARED_S3}")


Uploading /home/stan/Git/micropattern/data/GSE245998/processed/gse245998_heemskerk_multivi_ready.h5mu -> s3://stan-sequencing-data/processed-active/gse245998-tbxt-multiome-integration/input/gse245998_heemskerk_multivi_ready.h5mu
Upload complete: s3://stan-sequencing-data/processed-active/gse245998-tbxt-multiome-integration/input/gse245998_heemskerk_multivi_ready.h5mu
